In [ ]:
pip install PyOpenGL PyOpenGL_accelerate

First, we'll import the necessary libraries, including `OpenGL` and `GLUT` for window management.

In [ ]:
from OpenGL.GL import *
from OpenGL.GLUT import *
from OpenGL.GLU import *

# Basic vertex shader source code
vertex_shader_source = """
#version 330 core
layout (location = 0) in vec3 aPos;
void main()
{
    gl_Position = vec4(aPos.x, aPos.y, aPos.z, 1.0);
}
"""

# Basic fragment shader source code (red color)
fragment_shader_source = """
#version 330 core
out vec4 FragColor;
void main()
{
    FragColor = vec4(1.0f, 0.0f, 0.0f, 1.0f); // Red color
}
"""

shader_program = None

Next, we need functions to compile and link our shaders into a program. This program will then be used by OpenGL to render our graphics.

In [ ]:
def compile_shader(source, shader_type):
    shader = glCreateShader(shader_type)
    glShaderSource(shader, source)
    glCompileShader(shader)

    if not glGetShaderiv(shader, GL_COMPILE_STATUS):
        info_log = glGetShaderInfoLog(shader).decode('utf-8')
        print(f"Shader compilation failed:\n{info_log}")
        glDeleteShader(shader)
        return None
    return shader

def create_shader_program(vertex_source, fragment_source):
    vertex_shader = compile_shader(vertex_source, GL_VERTEX_SHADER)
    fragment_shader = compile_shader(fragment_source, GL_FRAGMENT_SHADER)

    if vertex_shader is None or fragment_shader is None:
        return None

    program = glCreateProgram()
    glAttachShader(program, vertex_shader)
    glAttachShader(program, fragment_shader)
    glLinkProgram(program)

    if not glGetProgramiv(program, GL_LINK_STATUS):
        info_log = glGetProgramInfoLog(program).decode('utf-8')
        print(f"Shader program linking failed:\n{info_log}")
        glDeleteProgram(program)
        return None

    glDeleteShader(vertex_shader)
    glDeleteShader(fragment_shader)
    return program

Now, let's define a function to initialize OpenGL and create our shader program. We'll also set up the vertex data for a simple triangle.

In [ ]:
import numpy as np

def init_opengl():
    global shader_program, VBO, VAO

    glClearColor(0.2, 0.3, 0.3, 1.0) # Set background color to dark teal

    shader_program = create_shader_program(vertex_shader_source, fragment_shader_source)
    if shader_program is None:
        print("Failed to create shader program. Exiting.")
        exit(1)

    # Vertex data for a triangle
    vertices = np.array([
        -0.5, -0.5, 0.0, # Left bottom
         0.5, -0.5, 0.0, # Right bottom
         0.0,  0.5, 0.0  # Top
    ], dtype=np.float32)

    # Create Vertex Array Object (VAO) and Vertex Buffer Object (VBO)
    VAO = glGenVertexArrays(1)
    VBO = glGenBuffers(1)

    glBindVertexArray(VAO)

    glBindBuffer(GL_ARRAY_BUFFER, VBO)
    glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)

    # Configure vertex attributes
    glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, 3 * ctypes.sizeof(GLfloat), ctypes.c_void_p(0))
    glEnableVertexAttribArray(0)

    glBindBuffer(GL_ARRAY_BUFFER, 0) # Unbind VBO
    glBindVertexArray(0) # Unbind VAO

Finally, we define the display function that will be called to draw frames, and the main function to set up the GLUT window and start the event loop.

It seems that `freeglut3-dev` (or a similar GLUT development package) is not installed on the system, which is required for `glutInit` and other GLUT functions to be available. We need to install it.

In [ ]:
!sudo apt-get update
!sudo apt-get install -y freeglut3-dev

Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 4,685 B in 1s (3,740 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [ ]:
def display():
    glClear(GL_COLOR_BUFFER_BIT)

    glUseProgram(shader_program)
    glBindVertexArray(VAO)
    glDrawArrays(GL_TRIANGLES, 0, 3)

    glutSwapBuffers()

def reshape(width, height):
    glViewport(0, 0, width, height)

def keyboard(key, x, y):
    if key == b'\x1b': # Escape key
        glutLeaveMainLoop()

def main():
    glutInit(sys.argv)
    glutInitDisplayMode(GLUT_RGBA | GLUT_DOUBLE)
    glutInitWindowSize(800, 600)
    glutCreateWindow(b"Simple OpenGL Triangle")

    init_opengl()

    glutDisplayFunc(display)
    glutReshapeFunc(reshape)
    glutKeyboardFunc(keyboard)

    print("OpenGL setup complete. A window should appear shortly.")
    print("Press ESC to close the window.")
    glutMainLoop()

import sys

# It's important to run this in a way that allows the GLUT main loop to run.
# In some environments (like certain notebooks), directly calling main() might block.
# For now, we'll provide the function and let the user execute it if they wish.
# If running in a script, you would simply call main()

# To run this example, execute the `main()` function.
# Note: If running in Colab, you might need a VNC server or similar to see the window.
# For simpler headless rendering or image output, other libraries might be more suitable.

# The direct call to main() is commented out to prevent session crashes in Colab.
# To run this code and see the OpenGL window, you would need to execute it in an
# environment with a graphical display server, or set up a VNC server for Colab.
# main()

In [ ]:
main()

### Step 1: Install VNC Server and Desktop Environment

We need to install a VNC server (like `tightvncserver`) and a minimal desktop environment (like `xfce4` or `xterm`). `xterm` is lighter and often sufficient for just running OpenGL applications.


In [ ]:
# Install necessary packages for VNC and a virtual display
!sudo apt-get update
!sudo apt-get install -y tightvncserver xterm x11vnc


Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 4,685 B in 1s (4,674 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

### Step 2: Set VNC Password

You'll be prompted to set a password for your VNC server. This password will be used to connect to your VNC session from your local machine. You can also choose to set a view-only password (optional).


In [ ]:
# Set VNC password - you will be prompted to enter a password in the console
!tightvncserver :1


tightvncserver: The USER environment variable is not set.


### Step 3: Configure and Start VNC Server

Now we'll kill the previous VNC server (if it's running) and start a new one with a virtual display (using `Xvfb`) and configure `xterm` to run inside it. This setup will create a virtual display and a VNC server listening on port 5901 (for display `:1`).

In [ ]:
import subprocess
import os
import time

# Kill any existing VNC server on display :1
!tightvncserver -kill :1 > /dev/null 2>&1

# Remove old log file if it exists
!rm -f ~/.vnc/*.log

# Start a virtual display (Xvfb) and a VNC server (x11vnc) and keep it running in the background
# This command sets up the VNC server to listen on port 5900 + display_number (5901 for :1)
# We'll use a simple xterm as the desktop environment
# Replace 'your_vnc_password' with the password you set in the previous step.
# You can also use a command like `echo "your_vnc_password" | vncpasswd -f > ~/.vnc/passwd` to set it programmatically

# Create a simple xstartup file if it doesn't exist
xstartup_content = '''#!/bin/bash
xterm -geometry 80x24+10+10 -ls -title "My VNC Desktop" &
'''

with open(os.path.expanduser('~/.vnc/xstartup'), 'w') as f:
    f.write(xstartup_content)
!chmod +x ~/.vnc/xstartup

# Start tightvncserver with display :1. This will use the xstartup script.
print("Starting TightVNC server on :1...")
process = subprocess.Popen(['tightvncserver', ':1', '-geometry', '1280x800', '-depth', '24', '-rfbport', '5901'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait a moment for the server to start
time.sleep(5)

# Check if the process is still running and report
if process.poll() is None:
    print("TightVNC server started successfully on display :1 (port 5901).")
    print("You can connect to this Colab instance using a VNC client at port 5901.")
    print("Your OpenGL applications will run within this VNC session.")
else:
    stdout, stderr = process.communicate()
    print(f"Failed to start TightVNC server. Exit code: {process.returncode}")
    print("STDOUT:", stdout.decode())
    print("STDERR:", stderr.decode())

# You can also use ngrok to expose this port to the internet if you want to connect from outside Colab directly.
# This is more advanced and requires an ngrok account and authentication token.


FileNotFoundError: [Errno 2] No such file or directory: '/root/.vnc/xstartup'

### Step 4: Connect to the VNC Server

To connect to this VNC server:

1.  **Port Forwarding**: You'll need to use a tool for port forwarding, such as `ssh -L 5901:localhost:5901 user@colab_runtime_ip` (if you can get the Colab runtime IP), or more commonly, a service like **ngrok** to expose the VNC port (5901) to the internet.

    *   **Using ngrok (recommended for simplicity):**
        ```bash
        # Install ngrok
        !wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
        !unzip ngrok-stable-linux-amd64.zip

        # Authenticate with your ngrok auth token (replace <YOUR_NGROK_AUTH_TOKEN>)
        # Get your token from https://dashboard.ngrok.com/auth/your-authtoken
        !./ngrok authtoken <YOUR_NGROK_AUTH_TOKEN>

        # Start ngrok tunnel for VNC (port 5901) in the background
        !./ngrok tcp 5901 &

        # Get the ngrok public URL (this might take a few seconds)
        import time
        time.sleep(5)
        import requests
        res = requests.get('http://localhost:4040/api/tunnels')
        vnc_tunnel_url = res.json()['tunnels'][0]['public_url']
        print(f"VNC Tunnel URL (connect your VNC client to this address): {vnc_tunnel_url}")
        ```

2.  **VNC Client**: Download and install a VNC client on your local machine (e.g., RealVNC Viewer, TightVNC Viewer, TigerVNC). In the VNC client, use the address provided by ngrok (e.g., `tcp://0.tcp.ngrok.io:XXXXX`) and the VNC password you set earlier.

Once connected, you will see an `xterm` window. You can then run your Python OpenGL script (e.g., `python -c "from OpenGL.GLUT import *; from OpenGL.GLU import *; from OpenGL.GL import *; import sys; display(); reshape(800, 600); keyboard(b'\x1b', 0, 0); main()"`) from within that `xterm` to see the OpenGL window.